# NB14：企業案例 — 企業內部員工 AI 助理系統設計

**系列：** 台灣科技公司 RAG/AI 實戰學習筆記  
**難度：** ⭐⭐⭐⭐⭐  
**預計時間：** 120 分鐘  

---

## 🎯 學習目標

本案例以一家 **500 人台灣科技公司** 為背景，設計完整的企業內部員工 AI 助理系統，涵蓋：

- 角色存取控制（RBAC）在內部 AI 系統的設計與實作
- 多來源知識庫整合（HR 文件、IT 文件、流程文件）
- 機密資料保護：確保員工只能查詢授權範圍內的資料
- 個人生產力功能：會議記錄摘要、Email 草稿協助
- 內部工具整合：模擬 Jira、Slack、HR 系統
- 新人入職助理與成本優化策略


---
## Part 0：系統需求分析

### 0.1 內部 vs 外部聊天機器人的差異

| 面向 | 外部客服機器人 | 內部員工助理 |
|------|--------------|-------------|
| 使用者信任度 | 低（匿名用戶） | 高（已驗證員工） |
| 資料敏感程度 | 公開產品資訊 | 薪資、人事、商業機密 |
| 使用模式 | 高頻、短問題 | 低頻、深度查詢 |
| 身份驗證 | 非必要 | SSO / AD 整合必須 |
| 回答風格 | 正式、品牌一致 | 口語、同事感 |
| 成本壓力 | 每次對話需控制成本 | 可接受較高單次成本換取效率 |
| 失敗容忍度 | 低（影響客戶體驗） | 中（員工會回報問題）|

### 0.2 系統架構圖

```
員工請求
    │
    ▼
SSO 身份驗證（AD / Google Workspace）
    │
    ▼
RBAC 過濾層（角色 × 部門 × 機密等級）
    │
    ├─────────────────────────────────┐
    ▼                                 ▼
意圖分類器                        工具路由器
    │                                 │
    ├── HR RAG（人事政策）         ├── Mock Jira
    ├── IT RAG（技術支援）         ├── Mock Slack
    └── Process RAG（流程文件）    └── Mock HR System
    │
    ▼
回應生成（GPT-4o-mini）
    │
    ▼
使用者回應 + 使用記錄
```

### 0.3 使用者角色說明

| 角色 | 存取等級 | 可查詢內容 |
|------|---------|----------|
| 新進員工（new_hire） | 1 | 入職指南、公開政策、IT 基礎設定 |
| 工程師（engineer） | 2 | + 技術文件、程式碼審查規範、IT 支援 |
| 主管（manager） | 3 | + 部門 KPI 報告、薪酬範圍、考績政策 |
| HR 人員（hr_staff） | 4 | + 所有人事資料、薪資結構、離職記錄 |
| IT 管理員（it_admin） | 5 | + 系統架構、安全政策、所有技術文件 |


In [ ]:
# 環境設定
import os
import json
import uuid
import random
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
from dotenv import load_dotenv
import openai
import chromadb

load_dotenv()
client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

EMBED_MODEL = 'text-embedding-3-small'
CHAT_MODEL  = 'gpt-4o-mini'

def embed(texts: List[str]) -> List[List[float]]:
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [r.embedding for r in resp.data]

def chat(messages: List[Dict], temperature=0.3, max_tokens=1024) -> str:
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return resp.choices[0].message.content

print('✅ 環境設定完成')
print(f'   Chat Model:  {CHAT_MODEL}')
print(f'   Embed Model: {EMBED_MODEL}')


---
## Part 1：多來源知識庫建立（Multi-source Knowledge Base）

建立三個獨立的 ChromaDB 集合：
- `hr_docs`：人事政策（15 份文件）
- `it_docs`：IT 支援文件（10 份文件）
- `process_docs`：流程文件（10 份文件）

每份文件都包含元資料：`department`、`sensitivity_level`、`doc_owner`


In [ ]:
# HR 文件資料
HR_DOCS = [
    {'id': 'hr_001', 'title': '年假與特休政策',
     'content': '員工依照年資享有特休假：任職滿6個月可請3天、滿1年7天、滿2年10天、滿3年14天、滿5年15天。特休假應於年度結束前使用完畢，未使用部分依勞基法規定折算工資。請假需提前3個工作天提出申請，並填寫人資系統中的請假單。緊急事假可事後補填，但需主管簽核。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'internal', 'doc_owner': 'hr_team', 'doc_type': 'policy'}},
    {'id': 'hr_002', 'title': '差旅與費用報銷政策',
     'content': '出差申請須於出發前5個工作天提交，經直屬主管及部門主管雙重核准。國內出差日支費用上限：住宿每晚新台幣3,000元、餐費每日600元、交通費實報實銷。國際出差依目的地分級：A級（美日歐）住宿每晚USD 200、B級（東南亞）USD 120。費用報銷需附原始收據，出差返回後10個工作天內完成申報，逾期恕不受理。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'internal', 'doc_owner': 'hr_team', 'doc_type': 'policy'}},
    {'id': 'hr_003', 'title': '薪資結構與調薪政策',
     'content': '公司薪資結構分為本薪、績效獎金、年終獎金三部分。本薪依職級帶（Band）決定，Band 1-3為初階、Band 4-6為中階、Band 7-9為高階。年度調薪通常在每年3月執行，調幅依個人績效評等決定：A（傑出）8-12%、B（良好）4-7%、C（達標）0-3%、D（待改善）不調薪。此文件僅供HR人員及主管參閱。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'confidential', 'doc_owner': 'hr_director', 'doc_type': 'policy'}},
    {'id': 'hr_004', 'title': '員工福利總覽',
     'content': '公司提供以下福利：1) 健康保險：全額補助員工及眷屬健保費。2) 員工健檢：每年補助5,000元健康檢查費用。3) 教育訓練補助：每年12,000元外部課程補助，需於完訓後3個月內申報。4) 員工餐廳：提供每日午餐補助100元。5) 生日假：生日當月可休半天帶薪假。6) 三節禮金：春節、端午、中秋各發放禮券2,000元。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'public', 'doc_owner': 'hr_team', 'doc_type': 'benefit'}},
    {'id': 'hr_005', 'title': '工作規則與行為準則',
     'content': '員工應遵守公司工作規則：上班時間為週一至週五09:00-18:00，午休12:00-13:00。實施彈性上班制，核心時間10:00-16:00須在崗。禁止行為包括：洩露公司機密、利用職務之便謀取私利、職場霸凌及性騷擾。違反行為準則將依情節輕重處以書面警告、停職或解僱。所有員工須每年完成行為準則線上訓練並簽署承諾書。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'public', 'doc_owner': 'legal_team', 'doc_type': 'policy'}},
    {'id': 'hr_006', 'title': '績效考核制度',
     'content': '公司採用半年度績效考核制度，分別於6月及12月執行。考核流程：員工自評→主管評分→校準會議→結果公布。評分維度：目標達成率（50%）、工作能力（30%）、團隊合作（20%）。360度回饋每年執行一次，結果供發展參考不直接影響考績。主管不得對直屬員工給予連續兩年C以下評等，如有需配合績效改善計劃（PIP）。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'internal', 'doc_owner': 'hr_team', 'doc_type': 'policy'}},
    {'id': 'hr_007', 'title': '育嬰假與育兒政策',
     'content': '依勞基法及性別工作平等法，員工享有以下育兒相關假別：產假56天（含例假）、陪產假7天、育嬰留停最長2年。公司額外提供：孕婦彈性上下班制、哺乳室使用、0-6歲子女托育補助每月3,000元。復職後保障原職或同等職位，薪資不得低於離職前水準。申請育嬰假需提前30天提出書面申請。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'internal', 'doc_owner': 'hr_team', 'doc_type': 'policy'}},
    {'id': 'hr_008', 'title': '離職流程與交接規範',
     'content': '員工離職應依合約規定提前通知（試用期7天、正式員工30天、主管級60天）。離職流程：1) HR系統提交離職申請。2) 與主管確認最後上班日。3) 執行工作交接（系統帳號、文件、客戶聯絡人）。4) 歸還公司財物（門禁卡、電腦、手機）。5) HR辦理人事手續。6) 核發服務年資證明。離職員工簽署保密協議效力持續2年，競業禁止條款依個別合約規定。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'internal', 'doc_owner': 'hr_team', 'doc_type': 'process'}},
    {'id': 'hr_009', 'title': '遠端工作政策',
     'content': '公司實施混合辦公制度（Hybrid Work）：每週至少3天在辦公室、最多2天遠端。遠端工作資格：試用期滿且工作表現達標。申請方式：透過HR系統提交遠端工作計劃，需主管同意。遠端工作期間仍需在核心時間內回應訊息，並參加所有線上會議。公司提供居家辦公補助：每月500元網路費補助、每年5,000元辦公設備購置補助。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'public', 'doc_owner': 'hr_team', 'doc_type': 'policy'}},
    {'id': 'hr_010', 'title': '員工申訴與仲裁程序',
     'content': '員工如對工作環境、主管行為或公司決策有異議，可透過以下管道申訴：1) 直接與HR面談（保密處理）。2) 匿名申訴信箱（每季由獨立委員會審查）。3) 員工關係熱線（分機8888）。申訴處理時程：收件後5個工作天確認受理、20個工作天完成調查、30個工作天提出處理結果。公司承諾不對申訴員工進行報復，違者依規定嚴處。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'public', 'doc_owner': 'legal_team', 'doc_type': 'process'}},
    {'id': 'hr_011', 'title': '新人入職清單',
     'content': '新進員工報到第一週完成事項：Day 1：領取設備（筆電、門禁卡、名片）、完成基本帳號設定（Email、Slack、Jira）、參加歡迎午餐。Day 2-3：完成必讀文件清單、觀看公司文化影片、與HR進行30分鐘orientation。Week 1：與直屬主管進行1-on-1確認3個月目標、拜訪相關部門同事、完成資安教育訓練。第一個月須完成：行為準則測驗、系統存取申請、導師（Mentor）配對。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'public', 'doc_owner': 'hr_team', 'doc_type': 'onboarding'}},
    {'id': 'hr_012', 'title': '薪資發放與扣繳說明',
     'content': '薪資於每月10日發放（遇假日提前）。薪資單可在HR系統下載，包含：本薪、加班費、各項補助、應扣項目（勞健保、所得稅預扣）。加班費計算：平日每小時為月薪÷240×1.33，假日×2。所得稅依個人薪資級距預扣，年度結算後多退少補。薪資相關問題請聯絡HR（分機8866）或薪資財務部門。薪資資訊屬個人機密，不得對外揭露。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'confidential', 'doc_owner': 'finance_hr', 'doc_type': 'policy'}},
    {'id': 'hr_013', 'title': '訓練與發展計劃',
     'content': '公司提供系統化的員工發展計劃：1) 內部講師制度：資深員工可申請成為內部講師，每場次補助2,000元。2) 線上學習平台：Coursera、Udemy企業帳號，每月最多20小時。3) 技術讀書會：每月一次，由工程師輪流分享。4) 管理培訓班：針對潛力主管，每季舉辦2天工作坊。5) 外派學習：年度最多2名員工獲選赴海外總部交流3個月。發展計劃需與主管共同制定個人IDP（Individual Development Plan）。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'public', 'doc_owner': 'hr_team', 'doc_type': 'benefit'}},
    {'id': 'hr_014', 'title': '員工持股計劃（ESOP）',
     'content': '公司提供員工持股計劃（ESOP）：任職滿1年後可申請參與。員工每月薪資提撥5%-15%購買公司股票，公司依提撥比例給予25%配對（Matching）。持股需鎖定2年後方可出售（Vesting Schedule）。ESOP委員會每季開會決定購股價格（通常為市價85折）。離職員工未vesting股份依規定處理。此計劃詳情屬機密，不得對外揭露，完整條款請洽HR主任。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'restricted', 'doc_owner': 'hr_director', 'doc_type': 'benefit'}},
    {'id': 'hr_015', 'title': '健康與安全政策',
     'content': '公司致力維護安全健康的工作環境。職業安全衛生管理：每季進行辦公室安全檢查、每年辦理消防演習。心理健康支援：提供EAP（員工協助方案），免費心理諮商每年6次。疫情/傳染病政策：員工出現發燒症狀應在家休息，依規定申請病假不扣薪。辦公室禁菸（含電子菸），吸菸者請至大樓外指定區域。工作相關傷病依勞工保險條例申請職災補償。',
     'metadata': {'department': 'HR', 'sensitivity_level': 'public', 'doc_owner': 'safety_team', 'doc_type': 'policy'}},
]

print(f'✅ HR 文件：{len(HR_DOCS)} 份')


In [ ]:
# IT 文件資料
IT_DOCS = [
    {'id': 'it_001', 'title': 'VPN 設定指南',
     'content': '公司使用 Cisco AnyConnect VPN。安裝步驟：1) 至 IT Portal (it.company.com) 下載 AnyConnect 安裝程式。2) 安裝後輸入伺服器位址：vpn.company.com。3) 使用公司 Email 帳號及密碼登入。4) 首次登入需完成 MFA 設定（推薦使用 Microsoft Authenticator）。連線問題排除：確認網路連線正常、嘗試切換 VPN Gateway（選項：Primary/Backup）、清除 AnyConnect 快取。IT支援：helpdesk@company.com 或分機5555。',
     'metadata': {'department': 'IT', 'sensitivity_level': 'internal', 'doc_owner': 'it_infra', 'doc_type': 'guide'}},
    {'id': 'it_002', 'title': '新員工設備申請流程',
     'content': '新進員工設備由HR通知IT部門事先準備。標準配備：MacBook Pro 14吋（工程師）或 MacBook Air（其他職位）、顯示器（24吋）、鍵盤滑鼠組。特殊需求（如雙螢幕、特規硬體）需主管簽核後申請。報到當天至IT室（B1）領取並完成設定。額外軟體安裝需透過 IT Portal 提交申請，IT審核後48小時內處理。個人設備（BYOD）使用規範請參閱資安政策文件。',
     'metadata': {'department': 'IT', 'sensitivity_level': 'internal', 'doc_owner': 'it_helpdesk', 'doc_type': 'process'}},
    {'id': 'it_003', 'title': '資訊安全政策',
     'content': '公司資訊安全基本要求：1) 密碼政策：最少12位、含大小寫字母數字及符號、每90天更換、不得重複使用前5次密碼。2) 螢幕鎖定：閒置5分鐘自動鎖定。3) 加密：公司電腦全磁碟加密（FileVault/BitLocker）。4) 資料分類：依機密等級（Public/Internal/Confidential/Restricted）處理。5) 禁止：使用個人Email傳送公司文件、將機密資料儲存於個人雲端空間。違反資安政策依嚴重性處理，最重可解僱並追究法律責任。',
     'metadata': {'department': 'IT', 'sensitivity_level': 'internal', 'doc_owner': 'ciso', 'doc_type': 'policy'}},
    {'id': 'it_004', 'title': '系統帳號申請與管理',
     'content': '員工帳號申請流程：所有系統存取須透過 IT Portal 提交申請，填寫：申請系統名稱、業務需求說明、資料存取範圍、主管電子核准。標準處理時間：一般系統3個工作天、核心系統（財務/人事）5個工作天且需CISO核准。帳號權限遵循最小權限原則（Principle of Least Privilege）。離職員工帳號於最後上班日24小時內停用。每季進行帳號存取審查（Access Review），不再需要的權限應主動申請移除。',
     'metadata': {'department': 'IT', 'sensitivity_level': 'internal', 'doc_owner': 'it_security', 'doc_type': 'process'}},
    {'id': 'it_005', 'title': '常見IT問題排除手冊',
     'content': '常見問題解決方法：Q1: 無法登入公司Email？→ 確認密碼是否到期，前往 account.company.com 重設。Q2: Slack 無法傳送訊息？→ 檢查網路連線，嘗試重新登入。Q3: 印表機無法使用？→ 確認已連接公司網路，重新安裝驅動程式（至 IT Portal 下載）。Q4: 電腦跑很慢？→ 關閉不必要的應用程式，確認儲存空間足夠（建議保留20%以上）。Q5: 忘記 MFA 設備？→ 聯絡 IT Helpdesk（分機5555）進行身份驗證後重設。',
     'metadata': {'department': 'IT', 'sensitivity_level': 'public', 'doc_owner': 'it_helpdesk', 'doc_type': 'guide'}},
    {'id': 'it_006', 'title': '雲端服務使用規範',
     'content': '公司核准使用之雲端服務：AWS（主要雲端平台）、Google Workspace（辦公軟體）、Slack（即時通訊）、Zoom（視訊會議）、GitHub Enterprise（程式碼管理）、Jira/Confluence（專案管理）。禁止使用個人帳號存取公司資料。AI工具使用規範：不得將機密資料輸入ChatGPT等公開AI服務，需使用公司核准的AI工具（清單詳見IT Portal）。SaaS採購需IT及財務部門共同審核，個人不得自行簽約。',
     'metadata': {'department': 'IT', 'sensitivity_level': 'internal', 'doc_owner': 'ciso', 'doc_type': 'policy'}},
    {'id': 'it_007', 'title': '事件通報與應變程序',
     'content': '資安事件分級：P1（嚴重）：系統全面中斷、資料外洩、勒索軟體，立即通報 CISO 及 IT 主管。P2（重要）：部分系統中斷、可疑存取行為，4小時內通報IT安全團隊。P3（一般）：單一設備問題、可疑郵件，透過 Jira 提交工單。發現可疑情況請立即：1) 中斷設備網路連線。2) 保留所有記錄。3) 撥打資安熱線（分機9999）。4) 不得自行嘗試修復。事件處理後需完成事後報告（Post-Incident Report）。',
     'metadata': {'department': 'IT', 'sensitivity_level': 'internal', 'doc_owner': 'it_security', 'doc_type': 'process'}},
    {'id': 'it_008', 'title': '開發環境標準設定',
     'content': '工程師開發環境標準：IDE：VS Code（推薦）或 JetBrains 系列（需申請授權）。版本控制：Git + GitHub Enterprise（禁止使用個人GitHub）。套件管理：Python使用poetry或pip+venv、Node.js使用npm或pnpm、禁止使用未審核的套件來源。程式碼品質：必須安裝 ESLint/Pylint，PR前需通過 CI/CD 掃描。機密管理：禁止在程式碼中hardcode API Key，使用AWS Secrets Manager或環境變數。Docker及K8s使用規範詳見DevOps Wiki。',
     'metadata': {'department': 'IT', 'sensitivity_level': 'internal', 'doc_owner': 'devops_team', 'doc_type': 'guide'}},
    {'id': 'it_009', 'title': '網路架構與存取規則',
     'content': '辦公室網路分區：Corp-WiFi（一般員工）、Secure-WiFi（研發部門，需憑證認證）、Guest-WiFi（訪客，僅限上網）。防火牆規則：預設拒絕所有對外連線，白名單制度。研發人員需額外申請特定Port開放。遠端存取必須透過VPN，不得使用未授權的代理服務。網路使用行為由DLP系統監控，異常行為將觸發自動告警。詳細網路拓撲圖及IP配置屬機密文件，僅IT管理員可存取。',
     'metadata': {'department': 'IT', 'sensitivity_level': 'restricted', 'doc_owner': 'it_infra', 'doc_type': 'technical'}},
    {'id': 'it_010', 'title': '軟體授權管理政策',
     'content': '公司採用集中式軟體授權管理。已採購軟體（員工可直接使用）：Microsoft 365、Adobe Creative Cloud（設計部門）、Zoom、Slack、GitHub Enterprise、JetBrains All Products。申請新軟體：在 IT Portal 提交需求→IT評估安全性→財務審核預算→採購→部署。禁止安裝未授權軟體，IT會定期掃描設備。發現盜版軟體將立即移除，情節嚴重依規處分。個人購買後申請報銷的軟體需事前取得主管和IT共同核准。',
     'metadata': {'department': 'IT', 'sensitivity_level': 'internal', 'doc_owner': 'it_helpdesk', 'doc_type': 'policy'}},
]

print(f'✅ IT 文件：{len(IT_DOCS)} 份')


In [ ]:
# 流程文件資料
PROCESS_DOCS = [
    {'id': 'proc_001', 'title': '程式碼審查（Code Review）流程',
     'content': 'PR提交規範：1) 每個PR聚焦單一功能或修復，不超過400行變更。2) PR描述需包含：目的說明、測試方式、相關Jira票號。3) 至少需2位Reviewer（其中1位為資深工程師）核准。4) CI/CD必須全數通過（單元測試、Linting、Security Scan）。審查標準：功能正確性、程式碼可讀性、效能影響、安全性考量。緊急修復（Hotfix）可簡化流程，但需在合併後24小時內補齊文件。一般PR目標在48小時內完成審查。',
     'metadata': {'department': 'Engineering', 'sensitivity_level': 'internal', 'doc_owner': 'eng_team', 'doc_type': 'process'}},
    {'id': 'proc_002', 'title': '敏捷開發與Sprint流程',
     'content': '公司採用Scrum框架，Sprint週期2週。標準會議：Sprint Planning（每Sprint開始，2小時）、Daily Standup（每日15分鐘，09:30）、Sprint Review（Sprint結束前，1小時）、Retrospective（Sprint結束，1小時）。Story Point估算使用費波那契數列（1/2/3/5/8/13）。Definition of Done：功能完成、測試覆蓋率>80%、文件更新、PR合併、QA驗收。產品Backlog由Product Owner維護，優先級每週調整。',
     'metadata': {'department': 'Engineering', 'sensitivity_level': 'internal', 'doc_owner': 'eng_manager', 'doc_type': 'process'}},
    {'id': 'proc_003', 'title': '產品發布流程（Release Process）',
     'content': '版本發布流程：1) Feature Freeze（Sprint結束前3天）。2) QA全回歸測試（2天）。3) UAT（User Acceptance Test，1天）。4) 發布審核會議（Release Review Meeting）。5) 正式發布（週二或週四下午15:00後，避免週五）。回滾計劃：每次發布需有Roll-back Plan，發布後30分鐘監控關鍵指標。重大版本發布需提前2週通知相關部門。Hotfix可縮短流程但需CTO核准。所有發布記錄於Confluence Release Notes頁面。',
     'metadata': {'department': 'Engineering', 'sensitivity_level': 'internal', 'doc_owner': 'devops_team', 'doc_type': 'process'}},
    {'id': 'proc_004', 'title': '客戶事件處理（Incident Response）',
     'content': '客戶端事件處理SLA：P0（服務全中斷）：15分鐘內響應、1小時內更新狀態、4小時內解決。P1（主要功能異常）：30分鐘響應、2小時更新、8小時解決。P2（部分功能異常）：2小時響應、24小時解決。P3（輕微問題）：下個工作日響應。事件處理步驟：偵測→確認→通知→分析→解決→事後報告。On-call輪值每2週一輪，補貼：平日待命每日300元、假日待命每日600元。',
     'metadata': {'department': 'Engineering', 'sensitivity_level': 'internal', 'doc_owner': 'sre_team', 'doc_type': 'process'}},
    {'id': 'proc_005', 'title': '會議效率準則',
     'content': '公司會議文化準則：召集人責任：提前24小時寄送會議邀請含議程、確認必要與會者（決策者）、準時開始準時結束。與會者責任：準時出席、保持專注（手機靜音）、積極參與討論。會議記錄：每次會議必須有人負責記錄，格式：決議事項、行動項目（負責人+截止日）。30分鐘以下無需記錄，1小時以上需正式會議紀錄並存至Confluence。禁止無議程會議、禁止週五下午14:00後安排非緊急會議。',
     'metadata': {'department': 'Operations', 'sensitivity_level': 'public', 'doc_owner': 'ops_team', 'doc_type': 'guideline'}},
    {'id': 'proc_006', 'title': '採購申請流程',
     'content': '採購流程依金額分級：10萬以下：部門主管核准→財務審核→採購執行。10-50萬：部門主管+VP核准→財務審核→詢價3家以上→採購執行。50萬以上：需董事會核准，採購委員會公開招標。緊急採購（5萬以下）：可先行採購後補件，需主管事後簽核。採購需求提交：Jira採購申請板，填寫品項、預算、業務需求、期望到貨日。所有採購合約需經法務審查。',
     'metadata': {'department': 'Operations', 'sensitivity_level': 'internal', 'doc_owner': 'finance_team', 'doc_type': 'process'}},
    {'id': 'proc_007', 'title': '客戶合約管理流程',
     'content': '合約簽署流程：業務完成商務條件談判→法務審查合約（3-5個工作天）→客戶確認修訂→產品確認技術條款→主管/VP/CEO（依合約金額）電子簽署→存入合約管理系統。合約重要資訊必須登錄：合約期間、金額、服務範圍、SLA條款、續約提醒日（到期前90天）。合約到期前3個月業務啟動續約討論。合約文件屬機密，不得對外分享。違約處理依合約條款及法律建議處理。',
     'metadata': {'department': 'Sales', 'sensitivity_level': 'confidential', 'doc_owner': 'legal_team', 'doc_type': 'process'}},
    {'id': 'proc_008', 'title': '預算申請與控管流程',
     'content': '年度預算規劃：每年10月開始下一年度預算規劃，11月底前完成部門預算提交，12月董事會核定，1月開始執行。季度預算調整：各部門可提交季度調整申請，需說明原因及影響，VP核准後財務執行。費用控管：每月財務提供預算執行報告（Budget vs Actual），超支10%需主動說明，超支20%需提改善計劃。部門主管對預算執行負責，年底列入績效評估項目。',
     'metadata': {'department': 'Finance', 'sensitivity_level': 'confidential', 'doc_owner': 'cfo', 'doc_type': 'process'}},
    {'id': 'proc_009', 'title': '知識管理與文件規範',
     'content': '公司知識管理平台：Confluence（內部文件）、GitHub Wiki（技術文件）、Google Drive（共用檔案）。文件撰寫規範：每份文件需有：標題、作者、建立日期、最後更新日期、文件版本、審核人。文件分類標籤：部門/主題/機密等級。定期維護：文件負責人每季審查確認是否需要更新，過時文件標記為Deprecated。命名規則：[部門代碼]-[類型]-[主題]-[版本]，例：ENG-PROC-CodeReview-v2.1。禁止在個人電腦儲存唯一的知識文件。',
     'metadata': {'department': 'Operations', 'sensitivity_level': 'internal', 'doc_owner': 'ops_team', 'doc_type': 'guideline'}},
    {'id': 'proc_010', 'title': '跨部門協作與溝通規範',
     'content': '跨部門協作原則：1) 明確的需求文件：提出跨部門需求時需附上PRD或需求說明（至少半頁A4）。2) 對等聯絡人：需求方指定一位窗口，接收方亦指定一位負責人。3) 時程協商：不得單方面設定跨部門截止日，需雙方確認。4) 升級機制：協作卡關時，先在工作層級嘗試解決，無法解決則升級至各自主管，一週內若仍無結論升級至VP。Slack使用規範：非緊急事項用頻道、緊急事項才直接tag、不在工作時間外傳送非緊急訊息。',
     'metadata': {'department': 'Operations', 'sensitivity_level': 'public', 'doc_owner': 'ops_team', 'doc_type': 'guideline'}},
]

print(f'✅ 流程文件：{len(PROCESS_DOCS)} 份')


In [ ]:
# 建立 ChromaDB 集合並嵌入文件
chroma_client = chromadb.Client()

def build_collection(docs: List[Dict], collection_name: str) -> chromadb.Collection:
    """建立 ChromaDB 集合並嵌入所有文件"""
    collection = chroma_client.get_or_create_collection(collection_name)
    texts = [f"{d['title']}\n{d['content']}" for d in docs]
    ids   = [d['id'] for d in docs]
    metas = [d['metadata'] for d in docs]
    batch_size = 5
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        all_embeddings.extend(embed(texts[i:i+batch_size]))
    collection.add(ids=ids, embeddings=all_embeddings, documents=texts, metadatas=metas)
    return collection

print('正在建立知識庫集合...')
hr_collection      = build_collection(HR_DOCS, 'hr_docs')
print(f'  ✅ hr_docs 建立完成（{hr_collection.count()} 份文件）')
it_collection      = build_collection(IT_DOCS, 'it_docs')
print(f'  ✅ it_docs 建立完成（{it_collection.count()} 份文件）')
process_collection = build_collection(PROCESS_DOCS, 'process_docs')
print(f'  ✅ process_docs 建立完成（{process_collection.count()} 份文件）')

COLLECTIONS = {'hr': hr_collection, 'it': it_collection, 'process': process_collection}
print('\n✅ 多來源知識庫建立完成！')


---
## Part 2：角色存取控制（RBAC）

設計員工類別與 RBAC 過濾器，確保員工只能查詢授權範圍內的資料。

### 機密等級對照表

| 等級 | 標籤 | 說明 | 可存取角色 |
|------|------|------|----------|
| 1 | public | 全公司公開 | 所有員工（含新人）|
| 2 | internal | 公司內部 | 正式員工 |
| 3 | confidential | 機密 | 主管、HR |
| 4 | restricted | 高度機密 | HR主任、IT管理員 |

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Set, Optional

# 機密等級映射
SENSITIVITY_LEVELS = {
    'public':       1,
    'internal':     2,
    'confidential': 3,
    'restricted':   4,
}

# 角色權限定義
ROLE_PERMISSIONS = {
    'new_hire': {
        'clearance_level': 1,
        'allowed_collections': ['hr', 'it'],
        'max_sensitivity': 'public',
    },
    'engineer': {
        'clearance_level': 2,
        'allowed_collections': ['hr', 'it', 'process'],
        'max_sensitivity': 'internal',
    },
    'manager': {
        'clearance_level': 3,
        'allowed_collections': ['hr', 'it', 'process'],
        'max_sensitivity': 'confidential',
    },
    'hr_staff': {
        'clearance_level': 4,
        'allowed_collections': ['hr', 'it', 'process'],
        'max_sensitivity': 'restricted',
    },
    'it_admin': {
        'clearance_level': 4,
        'allowed_collections': ['hr', 'it', 'process'],
        'max_sensitivity': 'restricted',
    },
}


@dataclass
class Employee:
    employee_id: str
    name: str
    role: str
    department: str
    email: str
    start_date: str
    manager_id: Optional[str] = None
    leave_balance: int = 10
    
    @property
    def permissions(self) -> Dict:
        return ROLE_PERMISSIONS.get(self.role, ROLE_PERMISSIONS['new_hire'])
    
    @property
    def clearance_level(self) -> int:
        return self.permissions['clearance_level']
    
    @property
    def max_sensitivity(self) -> str:
        return self.permissions['max_sensitivity']
    
    @property
    def allowed_collections(self) -> List[str]:
        return self.permissions['allowed_collections']
    
    def can_access(self, sensitivity_level: str) -> bool:
        user_level = self.clearance_level
        doc_level  = SENSITIVITY_LEVELS.get(sensitivity_level, 99)
        return user_level >= doc_level
    
    def __str__(self):
        return f'{self.name} [{self.role}] @ {self.department} (Level {self.clearance_level})'


# 測試員工資料
EMPLOYEES = {
    'E001': Employee('E001', '王小明', 'new_hire',  'Engineering', 'wang@co.tw', '2024-11-01', 'E005', 3),
    'E002': Employee('E002', '李大華', 'engineer',  'Engineering', 'lee@co.tw',  '2022-03-15', 'E005', 12),
    'E003': Employee('E003', '張美麗', 'manager',   'Engineering', 'chang@co.tw','2019-07-01', 'E006', 15),
    'E004': Employee('E004', '陳秀蘭', 'hr_staff',  'HR',          'chen@co.tw', '2020-01-10', 'E007', 14),
    'E005': Employee('E005', '林志偉', 'it_admin',  'IT',          'lin@co.tw',  '2018-05-20', 'E006', 18),
}

for eid, emp in EMPLOYEES.items():
    print(f'  {emp} | 可存取等級：≤{emp.max_sensitivity} | 集合：{emp.allowed_collections}')

In [ ]:
class RBACFilter:
    """角色存取控制過濾器"""
    
    def __init__(self, collections: Dict, employees: Dict):
        self.collections = collections
        self.employees   = employees
    
    def query(self, employee: Employee, query_text: str, n_results: int = 3) -> Dict:
        """依使用者權限查詢知識庫"""
        all_results = []
        queried_collections = []
        filtered_count = 0
        
        # 嵌入查詢
        q_embed = embed([query_text])[0]
        
        for coll_name in employee.allowed_collections:
            collection = self.collections[coll_name]
            results = collection.query(
                query_embeddings=[q_embed],
                n_results=min(n_results, collection.count()),
                include=['documents', 'metadatas', 'distances']
            )
            
            queried_collections.append(coll_name)
            
            for doc, meta, dist in zip(
                results['documents'][0],
                results['metadatas'][0],
                results['distances'][0]
            ):
                doc_sensitivity = meta.get('sensitivity_level', 'public')
                if employee.can_access(doc_sensitivity):
                    all_results.append({
                        'content':     doc,
                        'metadata':    meta,
                        'distance':    dist,
                        'collection':  coll_name,
                        'sensitivity': doc_sensitivity,
                    })
                else:
                    filtered_count += 1
        
        # 依相關度排序
        all_results.sort(key=lambda x: x['distance'])
        top_results = all_results[:n_results]
        
        return {
            'results':            top_results,
            'queried_collections': queried_collections,
            'filtered_count':     filtered_count,
            'employee':           str(employee),
        }
    
    def answer(self, employee: Employee, query_text: str) -> str:
        """RBAC 過濾後生成回答"""
        retrieval = self.query(employee, query_text)
        
        if not retrieval['results']:
            return f'抱歉，根據您的存取權限，找不到與「{query_text}」相關的資料。'
        
        context = '\n\n---\n\n'.join(
            f"[{r['collection'].upper()} | {r['sensitivity']}]\n{r['content'][:400]}"
            for r in retrieval['results']
        )
        
        messages = [
            {'role': 'system', 'content': f'你是公司內部員工 AI 助理。使用者身份：{employee.name}（{employee.role}）。請根據以下公司文件回答問題，如文件中無相關資訊請如實說明。'},
            {'role': 'user',   'content': f'公司文件：\n{context}\n\n問題：{query_text}'}
        ]
        return chat(messages)


rbac = RBACFilter(COLLECTIONS, EMPLOYEES)
print('✅ RBAC 過濾器初始化完成')

In [ ]:
# Demo：相同問題，不同角色得到不同結果
query = '公司薪資結構和調薪政策是什麼？'
print(f'查詢問題：{query}')
print('=' * 60)

test_roles = ['E001', 'E002', 'E003', 'E004']  # new_hire, engineer, manager, hr_staff

for eid in test_roles:
    emp = EMPLOYEES[eid]
    result = rbac.query(emp, query, n_results=3)
    
    print(f'\n👤 {emp}')
    print(f'   查詢集合：{result["queried_collections"]}')
    print(f'   找到文件：{len(result["results"])} 份（過濾掉 {result["filtered_count"]} 份）')
    
    for r in result['results']:
        title = r['content'].split('\n')[0][:40]
        print(f'   - [{r["sensitivity"]}] {title}')

In [ ]:
# 完整 RBAC 問答示範
print('=== RBAC 問答示範：薪資政策查詢 ===')
print()

# 工程師查詢（應看不到機密薪資文件）
emp_eng = EMPLOYEES['E002']
print(f'👤 {emp_eng} 詢問：公司薪資結構是什麼？')
print()
ans_eng = rbac.answer(emp_eng, '公司薪資結構是什麼？')
print(ans_eng)

print('\n' + '=' * 60 + '\n')

# HR人員查詢（應能看到完整薪資文件）
emp_hr = EMPLOYEES['E004']
print(f'👤 {emp_hr} 詢問：公司薪資結構是什麼？')
print()
ans_hr = rbac.answer(emp_hr, '公司薪資結構是什麼？')
print(ans_hr)

---
## Part 3：個人生產力功能

三個核心生產力工具：
1. **MeetingNoteSummarizer** — 會議記錄結構化摘要
2. **EmailDraftHelper** — 專業 Email 草稿生成
3. **TaskExtractor** — 從文件中提取待辦事項

In [ ]:
class MeetingNoteSummarizer:
    """會議記錄摘要器"""
    
    SYSTEM_PROMPT = """你是一位專業的會議記錄整理助理。
請將原始會議記錄整理成結構化格式，包含：
1. 會議基本資訊（日期、參與者、主持人）
2. 討論摘要（每個議題的重點）
3. 決議事項（明確的決定）
4. 行動項目（負責人、截止日期、具體任務）
5. 下次會議時間（如有提及）

請用繁體中文輸出，格式清晰易讀。"""
    
    def summarize(self, transcript: str, meeting_date: str = None) -> str:
        date_hint = f'（會議日期：{meeting_date}）' if meeting_date else ''
        messages = [
            {'role': 'system', 'content': self.SYSTEM_PROMPT},
            {'role': 'user',   'content': f'請整理以下會議記錄{date_hint}：\n\n{transcript}'}
        ]
        return chat(messages, temperature=0.2, max_tokens=1200)


# 測試用會議記錄
SAMPLE_TRANSCRIPT = """
2024年12月10日，Q4產品規劃會議，10:00-11:30，線上會議
參與者：張美麗（主持，工程主管）、李大華（後端工程師）、王小明（前端工程師新人）、陳志偉（產品經理）、林雅婷（QA）

張：今天主要討論Q1的開發計劃，還有一些技術債的處理方式。陳你先說說產品這邊的需求？

陳：好的，Q1我們主要要完成三個功能：一是用戶儀表板重設計，二是API效能優化，三是多語系支援。用戶儀表板是最高優先級，客戶那邊等很久了。

李：API優化我之前有做過評估，主要瓶頸在資料庫查詢，估計需要兩個Sprint，我可以負責這塊。

張：好，那儀表板誰來負責？王小明你這邊狀況怎樣，上手了嗎？

王：上手中，但我需要王雅婷協助Review。

張：好，那儀表板由王小明主導、林雅婷協助，目標1月底完成設計稿、2月底上線。多語系先放後面，Q2再說。

林：我這邊QA自動化測試框架還沒建好，這個季度能不能排優先級？

張：我理解，這個也很重要。這樣，林雅婷你寫一個提案，下週五前給我，我們再決定要不要這個季度做。

陳：還有一件事，客戶端那邊要求要有使用報表功能，大概中等複雜度，要不要這季做？

張：先不做，放Q2 backlog。Q1就聚焦這兩個功能。好，下次會議定在12月24日同一時間，主要確認Sprint 1的完成狀況。
"""

summarizer = MeetingNoteSummarizer()
summary = summarizer.summarize(SAMPLE_TRANSCRIPT, '2024-12-10')
print('=== 會議記錄摘要 ===')
print(summary)

In [ ]:
class EmailDraftHelper:
    """Email 草稿生成助理"""
    
    SYSTEM_PROMPT = """你是一位專業的商務Email寫作助理。
請根據使用者提供的情境和意圖，生成專業、清楚的Email草稿。
注意：
- 語氣適當（依情境選擇正式或親切）
- 結構清晰（問候→主旨→內文→結語→署名）
- 避免冗詞贅字
- 如指定語言則使用指定語言，預設使用繁體中文"""
    
    def draft(self, sender: Employee, context: str, intent: str, language: str = 'zh-TW') -> str:
        lang_note = '英文' if language == 'en' else '繁體中文'
        messages = [
            {'role': 'system', 'content': self.SYSTEM_PROMPT},
            {'role': 'user', 'content': (
                f'寄件人：{sender.name}（{sender.role}，{sender.department}部門）\n'
                f'語言：{lang_note}\n'
                f'情境：{context}\n'
                f'意圖/需求：{intent}\n\n'
                f'請生成Email草稿：'
            )}
        ]
        return chat(messages, temperature=0.4, max_tokens=800)


class TaskExtractor:
    """待辦事項提取器"""
    
    def extract(self, document: str, assignee: str = None) -> List[Dict]:
        """從文件中提取待辦事項"""
        assignee_hint = f'，特別關注指派給「{assignee}」的事項' if assignee else ''
        messages = [
            {'role': 'system', 'content': '你是一位任務提取助理，請從文件中找出所有待辦事項（TODO、行動項目、被指派的任務），以 JSON 格式輸出。格式：[{"task": "任務描述", "assignee": "負責人", "deadline": "截止日（如有）", "priority": "high/medium/low"}]'},
            {'role': 'user', 'content': f'請從以下文件提取待辦事項{assignee_hint}：\n\n{document}'}
        ]
        response = chat(messages, temperature=0.1, max_tokens=600)
        
        # 解析 JSON
        try:
            # 找到 JSON 部分
            start = response.find('[')
            end   = response.rfind(']') + 1
            if start >= 0 and end > start:
                tasks = json.loads(response[start:end])
                return tasks
        except:
            pass
        return [{'task': response, 'assignee': assignee or '未指定', 'deadline': '未指定', 'priority': 'medium'}]


email_helper = EmailDraftHelper()
task_extractor = TaskExtractor()
print('✅ 生產力工具初始化完成')

# Email 示範
emp_mgr = EMPLOYEES['E003']
email_draft = email_helper.draft(
    sender=emp_mgr,
    context='新人王小明下週一開始工作，需通知團隊',
    intent='發送新人歡迎通知給整個工程團隊，讓大家知道新人資訊和安排一個歡迎午餐'
)
print('=== Email 草稿示範 ===')
print(email_draft)

In [ ]:
# 任務提取示範
tasks = task_extractor.extract(SAMPLE_TRANSCRIPT, '林雅婷')
print('=== 任務提取結果 ===')
print(f'從會議記錄中提取到 {len(tasks)} 個任務：')
print()
for i, task in enumerate(tasks, 1):
    print(f'{i}. [{task.get("priority", "?").upper()}] {task.get("task", "")}')    
    print(f'   負責人：{task.get("assignee", "未指定")} | 截止：{task.get("deadline", "未指定")}')

---
## Part 4：內部工具整合（Mock Integrations）

模擬三個企業內部系統的 API 整合：
- **Mock Jira**：建立工單、查詢工單
- **Mock Slack**：發送頻道訊息
- **Mock HR System**：查詢員工資訊、假期餘額

In [ ]:
import uuid
from datetime import datetime
from typing import Optional

# ==================== Mock Jira Connector ====================
class MockJira:
    """模擬 Jira 工單系統"""
    
    def __init__(self):
        self.tickets = {}
        self.ticket_counter = 100
    
    def create_ticket(
        self,
        title: str,
        description: str,
        assignee: str,
        priority: str = 'Medium',
        ticket_type: str = 'Bug',
        project: str = 'TECH'
    ) -> Dict:
        self.ticket_counter += 1
        ticket_id = f'{project}-{self.ticket_counter}'
        ticket = {
            'id':          ticket_id,
            'title':       title,
            'description': description,
            'assignee':    assignee,
            'priority':    priority,
            'type':        ticket_type,
            'status':      'Open',
            'created_at':  datetime.now().strftime('%Y-%m-%d %H:%M'),
            'url':         f'https://jira.company.com/browse/{ticket_id}'
        }
        self.tickets[ticket_id] = ticket
        return {'success': True, 'ticket': ticket}
    
    def get_ticket(self, ticket_id: str) -> Optional[Dict]:
        return self.tickets.get(ticket_id)
    
    def list_tickets(self, assignee: str = None) -> List[Dict]:
        tickets = list(self.tickets.values())
        if assignee:
            tickets = [t for t in tickets if t['assignee'] == assignee]
        return tickets


# ==================== Mock Slack Notifier ====================
class MockSlack:
    """模擬 Slack 訊息系統"""
    
    def __init__(self):
        self.messages = []
        self.channels = {
            '#general': '全公司',
            '#engineering': '工程團隊',
            '#hr-announcements': 'HR公告',
            '#it-support': 'IT支援',
            '#random': '閒聊',
        }
    
    def send_message(self, channel: str, message: str, sender: str = 'AI助理') -> Dict:
        if channel not in self.channels:
            return {'success': False, 'error': f'找不到頻道 {channel}'}
        
        msg = {
            'id':        str(uuid.uuid4())[:8],
            'channel':   channel,
            'sender':    sender,
            'message':   message,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        }
        self.messages.append(msg)
        print(f'  📨 [Slack] {channel} ← {sender}: {message[:60]}...' if len(message) > 60 else f'  📨 [Slack] {channel} ← {sender}: {message}')
        return {'success': True, 'message': msg}
    
    def get_channel_history(self, channel: str, limit: int = 5) -> List[Dict]:
        return [m for m in self.messages if m['channel'] == channel][-limit:]


# ==================== Mock HR System ====================
class MockHRSystem:
    """模擬 HR 人事系統"""
    
    def __init__(self, employees: Dict):
        self.employees = employees
    
    def get_employee_info(self, employee_id: str) -> Optional[Dict]:
        emp = self.employees.get(employee_id)
        if not emp:
            return None
        manager = self.employees.get(emp.manager_id)
        return {
            'employee_id':   emp.employee_id,
            'name':          emp.name,
            'role':          emp.role,
            'department':    emp.department,
            'email':         emp.email,
            'start_date':    emp.start_date,
            'manager_name':  manager.name if manager else '未設定',
            'leave_balance': emp.leave_balance,
        }
    
    def check_leave_balance(self, employee_id: str) -> Dict:
        emp = self.employees.get(employee_id)
        if not emp:
            return {'success': False, 'error': '員工不存在'}
        return {
            'success':       True,
            'employee_id':   employee_id,
            'name':          emp.name,
            'leave_balance': emp.leave_balance,
            'leave_used':    15 - emp.leave_balance,
        }


# 初始化所有模擬工具
jira_client = MockJira()
slack_client = MockSlack()
hr_system    = MockHRSystem(EMPLOYEES)

print('✅ 內部工具模擬系統初始化完成')
print(f'   Jira：已連線（{len(jira_client.tickets)} 個工單）')
print(f'   Slack：已連線（{len(slack_client.channels)} 個頻道）')
print(f'   HR System：已連線（{len(hr_system.employees)} 位員工）')

In [ ]:
# ==================== 工具整合 Agent ====================
class InternalToolAgent:
    """整合內部工具的 AI Agent"""
    
    TOOLS = [
        {
            'type': 'function',
            'function': {
                'name': 'create_jira_ticket',
                'description': '建立 Jira 工單（Bug報告、功能需求、IT支援等）',
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'title':       {'type': 'string', 'description': '工單標題'},
                        'description': {'type': 'string', 'description': '詳細描述'},
                        'assignee':    {'type': 'string', 'description': '指派給誰'},
                        'priority':    {'type': 'string', 'enum': ['Critical', 'High', 'Medium', 'Low']},
                        'ticket_type': {'type': 'string', 'enum': ['Bug', 'Feature', 'Task', 'IT-Support']},
                    },
                    'required': ['title', 'description', 'assignee', 'priority', 'ticket_type']
                }
            }
        },
        {
            'type': 'function',
            'function': {
                'name': 'send_slack_message',
                'description': '發送 Slack 訊息到指定頻道',
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'channel': {'type': 'string', 'description': '頻道名稱（如 #engineering）'},
                        'message': {'type': 'string', 'description': '訊息內容'},
                    },
                    'required': ['channel', 'message']
                }
            }
        },
        {
            'type': 'function',
            'function': {
                'name': 'get_employee_info',
                'description': '查詢員工基本資訊（姓名、部門、主管、剩餘假期）',
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'employee_id': {'type': 'string', 'description': '員工ID（如 E001）'},
                    },
                    'required': ['employee_id']
                }
            }
        },
    ]
    
    def __init__(self, jira, slack, hr):
        self.jira  = jira
        self.slack = slack
        self.hr    = hr
    
    def execute_tool(self, tool_name: str, args: Dict, caller: Employee) -> str:
        """執行工具並回傳結果"""
        if tool_name == 'create_jira_ticket':
            result = self.jira.create_ticket(**args)
            if result['success']:
                t = result['ticket']
                return f'成功建立工單 {t["id"]}：{t["title"]}\n連結：{t["url"]}'
            return f'建立工單失敗'
        
        elif tool_name == 'send_slack_message':
            result = self.slack.send_message(
                args['channel'], args['message'], sender=caller.name
            )
            return '訊息發送成功' if result['success'] else f'發送失敗：{result.get("error")}'
        
        elif tool_name == 'get_employee_info':
            info = self.hr.get_employee_info(args['employee_id'])
            if info:
                return json.dumps(info, ensure_ascii=False)
            return '找不到員工資訊'
        
        return f'未知工具：{tool_name}'
    
    def run(self, employee: Employee, user_request: str) -> str:
        """執行 Agent 對話迴圈"""
        messages = [
            {'role': 'system', 'content': f'你是公司內部員工 AI 助理。使用者：{employee.name}（{employee.role}）。請協助完成使用者的需求，必要時使用提供的工具。'},
            {'role': 'user',   'content': user_request}
        ]
        
        print(f'🤖 Agent 開始處理：{user_request[:50]}...' if len(user_request) > 50 else f'🤖 Agent 開始處理：{user_request}')
        
        for step in range(5):  # 最多5步
            resp = client.chat.completions.create(
                model=CHAT_MODEL,
                messages=messages,
                tools=self.TOOLS,
                tool_choice='auto'
            )
            msg = resp.choices[0].message
            
            if not msg.tool_calls:
                print(f'\n✅ Agent 完成（{step+1} 步）')
                return msg.content
            
            # 執行工具
            messages.append({'role': 'assistant', 'content': msg.content, 'tool_calls': [
                {'id': tc.id, 'type': 'function', 'function': {'name': tc.function.name, 'arguments': tc.function.arguments}}
                for tc in msg.tool_calls
            ]})
            
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f'  🔧 呼叫工具：{tc.function.name}({list(args.keys())})')
                result = self.execute_tool(tc.function.name, args, employee)
                messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
        
        return '處理超時，請稍後再試'


tool_agent = InternalToolAgent(jira_client, slack_client, hr_system)

# 示範：幫我建立 Bug 工單並通知團隊
emp_eng = EMPLOYEES['E002']
result = tool_agent.run(
    emp_eng,
    '我發現登入頁面在 Safari 上有顯示問題，按鈕點了沒反應。請幫我建立一個高優先級的 Bug 工單給 IT 支援，並在 #engineering 頻道通知大家。'
)
print('\n--- Agent 最終回覆 ---')
print(result)

---
## Part 5：新人入職助理（Onboarding Bot）

設計一個專為新進員工服務的入職助理，具備：
- 根據職位與入職日期生成個人化入職計劃
- 利用內部知識庫回答入職相關問題
- 追蹤入職進度（Session State）
- 5 輪對話示範

In [ ]:
class OnboardingAgent:
    """新人入職助理"""
    
    ONBOARDING_STEPS = {
        'engineer': [
            {'id': 'step_01', 'name': '領取設備（筆電、門禁卡）',         'day': 1,  'done': False},
            {'id': 'step_02', 'name': '設定公司Email帳號',                'day': 1,  'done': False},
            {'id': 'step_03', 'name': '安裝 VPN 並測試連線',              'day': 1,  'done': False},
            {'id': 'step_04', 'name': '參加 HR Orientation',             'day': 2,  'done': False},
            {'id': 'step_05', 'name': '閱讀資訊安全政策',                 'day': 2,  'done': False},
            {'id': 'step_06', 'name': '完成行為準則線上測驗',              'day': 3,  'done': False},
            {'id': 'step_07', 'name': '申請開發系統存取（GitHub、Jira）',  'day': 3,  'done': False},
            {'id': 'step_08', 'name': '與主管進行首次 1-on-1',            'day': 5,  'done': False},
            {'id': 'step_09', 'name': '完成首個 Code Review',             'day': 10, 'done': False},
            {'id': 'step_10', 'name': '配對導師（Mentor）並首次會面',      'day': 7,  'done': False},
        ],
        'default': [
            {'id': 'step_01', 'name': '領取設備與門禁卡',                  'day': 1,  'done': False},
            {'id': 'step_02', 'name': '設定公司Email帳號',                'day': 1,  'done': False},
            {'id': 'step_03', 'name': '參加 HR Orientation',             'day': 2,  'done': False},
            {'id': 'step_04', 'name': '閱讀公司政策文件',                  'day': 2,  'done': False},
            {'id': 'step_05', 'name': '完成行為準則線上測驗',              'day': 3,  'done': False},
            {'id': 'step_06', 'name': '與主管進行首次 1-on-1',            'day': 5,  'done': False},
            {'id': 'step_07', 'name': '認識部門同事',                     'day': 7,  'done': False},
        ],
    }
    
    def __init__(self, rbac_filter: 'RBACFilter'):
        self.rbac    = rbac_filter
        self.sessions: Dict[str, Dict] = {}
    
    def start_session(self, employee: Employee) -> str:
        """開始新的入職對話"""
        import copy
        steps_template = self.ONBOARDING_STEPS.get(employee.role, self.ONBOARDING_STEPS['default'])
        
        self.sessions[employee.employee_id] = {
            'employee':  employee,
            'steps':     copy.deepcopy(steps_template),
            'history':   [],
            'started_at': datetime.now().isoformat(),
        }
        
        # 生成個人化歡迎訊息
        step_names = [s['name'] for s in steps_template]
        welcome = chat([
            {'role': 'system', 'content': '你是友善的公司入職助理，請用溫暖親切的語氣歡迎新員工，並說明入職流程。'},
            {'role': 'user', 'content': (
                f'新員工資訊：{employee.name}，職位：{employee.role}，部門：{employee.department}，入職日：{employee.start_date}\n'
                f'入職清單：{step_names}\n'
                f'請生成一段溫暖的歡迎詞和入職清單說明（200字以內）'
            )}
        ], temperature=0.5, max_tokens=400)
        
        self.sessions[employee.employee_id]['history'].append({'role': 'assistant', 'content': welcome})
        return welcome
    
    def get_progress(self, employee_id: str) -> Dict:
        """取得入職進度"""
        if employee_id not in self.sessions:
            return {}
        session = self.sessions[employee_id]
        steps = session['steps']
        done = [s for s in steps if s['done']]
        return {
            'total':    len(steps),
            'done':     len(done),
            'pending':  len(steps) - len(done),
            'percent':  round(len(done) / len(steps) * 100),
            'steps':    steps,
        }
    
    def mark_step_done(self, employee_id: str, step_id: str) -> bool:
        """標記步驟完成"""
        if employee_id not in self.sessions:
            return False
        for step in self.sessions[employee_id]['steps']:
            if step['id'] == step_id:
                step['done'] = True
                return True
        return False
    
    def chat(self, employee_id: str, user_message: str) -> str:
        """進行入職問答對話"""
        if employee_id not in self.sessions:
            return '請先呼叫 start_session() 開始入職流程'
        
        session  = self.sessions[employee_id]
        employee = session['employee']
        
        # 用 RBAC 查詢知識庫
        retrieval = self.rbac.query(employee, user_message, n_results=2)
        context = '\n\n'.join(
            r['content'][:300] for r in retrieval['results']
        ) if retrieval['results'] else '（無相關文件）'
        
        # 取得進度資訊
        progress = self.get_progress(employee_id)
        progress_str = f"入職進度：{progress['done']}/{progress['total']} ({progress['percent']}%)"
        
        # 建立對話歷史
        history = session['history'][-6:]  # 最後3輪
        messages = [
            {'role': 'system', 'content': (
                f'你是公司友善的入職助理。使用者：{employee.name}（{employee.role}，{employee.department}部門）。\n'
                f'{progress_str}\n'
                f'相關政策文件：\n{context}\n'
                f'請用親切的語氣回答，若不確定請建議聯絡HR（分機8866）。'
            )},
        ] + history + [{'role': 'user', 'content': user_message}]
        
        response = chat(messages, temperature=0.4, max_tokens=500)
        
        # 更新對話歷史
        session['history'].append({'role': 'user',      'content': user_message})
        session['history'].append({'role': 'assistant', 'content': response})
        
        return response


onboarding_bot = OnboardingAgent(rbac)
print('✅ 入職助理初始化完成')

In [ ]:
# 5 輪入職對話示範
new_hire = EMPLOYEES['E001']  # 王小明

print(f'=== 新人入職助理示範 ===')
print(f'新員工：{new_hire.name}（{new_hire.role}，{new_hire.department}部門）')
print('=' * 60)

# 開始對話
welcome = onboarding_bot.start_session(new_hire)
print(f'\n🤖 助理：\n{welcome}')

# 5 輪問答
questions = [
    '我第一天要做什麼？需要帶什麼嗎？',
    'VPN 怎麼設定？我是 Mac 用戶。',
    '請假規定是怎樣的？我試用期可以請假嗎？',
    '我想申請 GitHub 和 Jira 的帳號，流程是什麼？',
    '公司有什麼員工福利？有健身房嗎？',
]

for i, q in enumerate(questions, 1):
    print(f'\n--- 第 {i} 輪 ---')
    print(f'👤 {new_hire.name}：{q}')
    response = onboarding_bot.chat(new_hire.employee_id, q)
    print(f'🤖 助理：\n{response}')

# 標記幾個步驟完成
onboarding_bot.mark_step_done(new_hire.employee_id, 'step_01')
onboarding_bot.mark_step_done(new_hire.employee_id, 'step_02')
onboarding_bot.mark_step_done(new_hire.employee_id, 'step_03')

progress = onboarding_bot.get_progress(new_hire.employee_id)
print(f'\n📊 目前入職進度：{progress["done"]}/{progress["total"]} ({progress["percent"]}%)')
for step in progress['steps']:
    status = '✅' if step['done'] else '⬜'
    print(f'  {status} [Day {step["day"]:2d}] {step["name"]}')

---
## Part 6：成本與使用分析

### 內部 AI 的成本特性

| 指標 | 外部客服機器人 | 內部員工助理 |
|------|--------------|-------------|
| 日活躍用戶 | 數千~數萬 | 數十~數百 |
| 平均對話長度 | 短（3-5輪） | 長（5-15輪）|
| 文件上下文 | 少 | 多（長文件）|
| 可接受成本 | 每次 $0.01-$0.05 | 每次 $0.05-$0.20 |
| ROI 衡量 | 客服效率提升 | 員工時間節省 |

In [ ]:
import random
from datetime import datetime, timedelta

# ==================== 成本計算模型 ====================
class CostCalculator:
    """AI 使用成本計算器"""
    
    # OpenAI gpt-4o-mini 定價（2024年，美元/1M tokens）
    INPUT_COST_PER_1M  = 0.15   # $0.15 per 1M input tokens
    OUTPUT_COST_PER_1M = 0.60   # $0.60 per 1M output tokens
    EMBED_COST_PER_1M  = 0.02   # $0.02 per 1M embed tokens
    USD_TO_TWD         = 32.0   # 匯率
    
    def calculate_query_cost(
        self,
        input_tokens: int,
        output_tokens: int,
        embed_tokens: int = 500
    ) -> Dict:
        input_cost  = input_tokens  / 1_000_000 * self.INPUT_COST_PER_1M
        output_cost = output_tokens / 1_000_000 * self.OUTPUT_COST_PER_1M
        embed_cost  = embed_tokens  / 1_000_000 * self.EMBED_COST_PER_1M
        total_usd   = input_cost + output_cost + embed_cost
        
        return {
            'input_tokens':  input_tokens,
            'output_tokens': output_tokens,
            'input_cost_usd':  round(input_cost, 6),
            'output_cost_usd': round(output_cost, 6),
            'embed_cost_usd':  round(embed_cost, 6),
            'total_usd':       round(total_usd, 6),
            'total_twd':       round(total_usd * self.USD_TO_TWD, 4),
        }
    
    def monthly_cost_per_employee(
        self,
        queries_per_day: float = 5,
        avg_input_tokens: int = 2000,
        avg_output_tokens: int = 500,
        working_days: int = 22
    ) -> Dict:
        monthly_queries = queries_per_day * working_days
        per_query_cost  = self.calculate_query_cost(avg_input_tokens, avg_output_tokens)
        monthly_usd     = per_query_cost['total_usd'] * monthly_queries
        
        return {
            'monthly_queries':     int(monthly_queries),
            'cost_per_query_usd':  per_query_cost['total_usd'],
            'monthly_cost_usd':    round(monthly_usd, 4),
            'monthly_cost_twd':    round(monthly_usd * self.USD_TO_TWD, 2),
            'annual_cost_twd':     round(monthly_usd * 12 * self.USD_TO_TWD, 2),
        }


calc = CostCalculator()

# 各角色使用模式
USAGE_PROFILES = {
    'new_hire':  {'queries_per_day': 8,  'avg_input': 1500, 'avg_output': 400},
    'engineer':  {'queries_per_day': 6,  'avg_input': 2500, 'avg_output': 600},
    'manager':   {'queries_per_day': 4,  'avg_input': 3000, 'avg_output': 700},
    'hr_staff':  {'queries_per_day': 10, 'avg_input': 2000, 'avg_output': 500},
    'it_admin':  {'queries_per_day': 7,  'avg_input': 3500, 'avg_output': 800},
}

# 公司員工分布（500人）
COMPANY_HEADCOUNT = {
    'new_hire':  20,
    'engineer':  280,
    'manager':   80,
    'hr_staff':  25,
    'it_admin':  15,
}

print('=== 每位員工月度成本分析 ===')
print(f'{"角色":<12} {"月查詢次數":<10} {"每次成本(USD)":<15} {"月費用(TWD)":<12} {"年費用(TWD)"}')
print('-' * 70)

total_monthly_twd = 0
role_costs = {}

for role, profile in USAGE_PROFILES.items():
    cost = calc.monthly_cost_per_employee(
        queries_per_day=profile['queries_per_day'],
        avg_input_tokens=profile['avg_input'],
        avg_output_tokens=profile['avg_output']
    )
    headcount = COMPANY_HEADCOUNT[role]
    dept_monthly_twd = cost['monthly_cost_twd'] * headcount
    total_monthly_twd += dept_monthly_twd
    role_costs[role] = {'per_person': cost, 'headcount': headcount, 'dept_total': dept_monthly_twd}
    
    print(f'{role:<12} {cost["monthly_queries"]:<10} ${cost["cost_per_query_usd"]:<14.5f} NT${cost["monthly_cost_twd"]:<11.2f} NT${cost["annual_cost_twd"]:.2f}')

print('-' * 70)
print(f'\n全公司月度總費用：NT$ {total_monthly_twd:,.2f}')
print(f'全公司年度總費用：NT$ {total_monthly_twd * 12:,.2f}')
print(f'每人平均月費用： NT$ {total_monthly_twd / 500:.2f}')

In [ ]:
# ==================== 使用分析模擬 ====================
# 模擬30天使用記錄
random.seed(42)

DEPARTMENTS = ['Engineering', 'HR', 'IT', 'Product', 'Sales', 'Finance']
TOPICS = ['請假政策', 'VPN設定', '費用報銷', '程式碼審查', '績效考核', '員工福利', '入職流程', '採購申請', '會議規範', '資安政策']

# 模擬使用記錄
usage_logs = []
for day in range(30):
    date = (datetime.now() - timedelta(days=30-day)).strftime('%Y-%m-%d')
    # 工作日才有使用
    if day % 7 in [5, 6]:  # 跳過週末
        continue
    for dept in DEPARTMENTS:
        # 各部門每天查詢次數（模擬）
        dept_queries = {
            'Engineering': random.randint(40, 80),
            'HR':          random.randint(20, 35),
            'IT':          random.randint(15, 30),
            'Product':     random.randint(25, 45),
            'Sales':       random.randint(10, 20),
            'Finance':     random.randint(8, 15),
        }
        for _ in range(dept_queries[dept]):
            usage_logs.append({
                'date':       date,
                'department': dept,
                'topic':      random.choice(TOPICS),
                'tokens':     random.randint(1500, 4000),
                'satisfied':  random.random() > 0.15,  # 85% 滿意度
            })

print(f'模擬使用記錄：{len(usage_logs)} 筆')

# 部門使用分析
dept_stats = {}
for log in usage_logs:
    dept = log['department']
    if dept not in dept_stats:
        dept_stats[dept] = {'queries': 0, 'tokens': 0, 'satisfied': 0}
    dept_stats[dept]['queries'] += 1
    dept_stats[dept]['tokens']  += log['tokens']
    if log['satisfied']:
        dept_stats[dept]['satisfied'] += 1

print('\n=== 部門使用分析（過去30天）===')
print(f'{"部門":<14} {"查詢次數":<10} {"平均Token":<12} {"滿意度"}')
print('-' * 50)
for dept, stats in sorted(dept_stats.items(), key=lambda x: -x[1]['queries']):
    avg_tok = stats['tokens'] // stats['queries']
    satisfaction = stats['satisfied'] / stats['queries'] * 100
    print(f'{dept:<14} {stats["queries"]:<10} {avg_tok:<12} {satisfaction:.1f}%')

# 熱門話題分析
topic_counts = {}
for log in usage_logs:
    topic_counts[log['topic']] = topic_counts.get(log['topic'], 0) + 1

print('\n=== 熱門查詢話題 TOP 5 ===')
for i, (topic, count) in enumerate(sorted(topic_counts.items(), key=lambda x: -x[1])[:5], 1):
    bar = '█' * (count // 30)
    print(f'{i}. {topic:<10} {bar} {count}次')

In [ ]:
# ==================== 預算分配建議 ====================
def generate_budget_recommendation(total_monthly_twd: float, employee_count: int) -> str:
    """生成預算建議報告"""
    per_person = total_monthly_twd / employee_count
    annual = total_monthly_twd * 12
    
    prompt = f"""你是一位企業 AI 成本顧問。請根據以下數據生成一份預算分配建議報告（繁體中文，300字以內）：

公司規模：{employee_count} 人
月度 AI 助理費用：NT$ {total_monthly_twd:,.0f}
年度 AI 助理費用：NT$ {annual:,.0f}
每人每月平均費用：NT$ {per_person:.2f}

請分析：
1. 費用是否合理（對比人力成本節省）
2. 成本優化建議（快取、模型分級、使用限制）
3. 預算分配優先順序
4. ROI 估算（假設每次查詢節省5分鐘人力）"""
    
    return chat([{'role': 'user', 'content': prompt}], temperature=0.3, max_tokens=600)


recommendation = generate_budget_recommendation(total_monthly_twd, 500)
print('=== AI 助理預算建議報告 ===')
print(recommendation)

# 成本優化策略
print('\n=== 成本優化策略 ===')
strategies = [
    ('語意快取（Semantic Cache）',  '對相似問題重用答案，預估降低 30-40% Token 消耗'),
    ('模型分級策略',               '簡單查詢用 gpt-4o-mini，複雜分析才用 gpt-4o，省 80% 費用'),
    ('上下文壓縮',                 '摘要長對話歷史，減少重複 Token，省 20-30%'),
    ('使用量配額',                 '每人每日查詢上限，防止濫用，確保資源公平分配'),
    ('預先嵌入文件',               '知識庫文件嵌入一次後快取，無需每次重新嵌入'),
]
for name, desc in strategies:
    print(f'  • {name}：{desc}')

---
## Part 7：面試 Q&A

以下是企業內部 AI 助理系統設計的常見面試題與解答參考。

### Q1：如何設計內部 AI 系統的 RBAC（角色存取控制）？

**重點回答：**

設計內部 AI 系統的 RBAC 需要考慮以下層次：

1. **身份驗證層**：與公司 SSO/LDAP/Active Directory 整合，確認使用者身份。

2. **角色定義層**：依業務需求定義角色（如工程師、主管、HR），每個角色對應一套存取權限，包含：
   - 可查詢的知識庫集合
   - 可存取的文件機密等級（public / internal / confidential / restricted）
   - 可使用的功能（基本查詢 vs. 薪資資料 vs. 系統管理）

3. **過濾執行層**：RAG 查詢時，在 retrieval 階段過濾掉使用者無權限的文件，而非在生成階段過濾（生成階段過濾有洩漏風險）。

4. **稽核記錄層**：所有查詢記錄需包含使用者ID、時間、查詢內容、回傳的文件ID，用於合規審查。

**關鍵原則：** 最小權限原則（Principle of Least Privilege）— 預設給予最低必要權限，需要更高權限時要有申請流程。

---

### Q2：如何確保機密資料不被未授權員工查詢到？

**重點回答：**

多層防護策略：

1. **Retrieval 前過濾**：在向量資料庫查詢時加入 metadata 過濾條件（`sensitivity_level <= user_clearance`），確保機密文件根本不進入 context。

2. **資料隔離**：將不同機密等級的文件存放在不同的 ChromaDB collection 或不同的向量資料庫實例，而非靠 metadata 過濾。

3. **系統提示加固**：在 system prompt 中明確告知 LLM 使用者的權限範圍，並要求不得推論或猜測未授權資訊。

4. **輸出監控**：對模型輸出進行 PII/機密資訊掃描（如正規表達式偵測薪資數字、身分證號等），觸發告警。

5. **定期審計**：定期審查 RBAC 規則，確保離職員工、職務調整的員工權限及時更新。

---

### Q3：多來源 RAG 系統如何決定要查詢哪個知識庫？

**重點回答：**

多來源 RAG 的路由策略有幾種：

1. **規則式路由（Rule-based）**：根據關鍵字或正規表達式判斷（如問到「請假」→ HR 庫）。速度快、可控，但不夠靈活。

2. **分類器路由（Classifier-based）**：訓練一個輕量分類模型（或用 embedding 相似度），判斷問題屬於哪個領域。

3. **並行查詢合併（Parallel Query）**：同時查詢所有可用知識庫，再依相關度分數合併排序。準確度高，但成本較高。

4. **LLM 路由（本案例）**：用 gpt-4o-mini 先判斷問題類型，再決定查哪個庫。靈活但增加一次 API 呼叫。

**建議**：對內部系統而言，並行查詢+RBAC過濾是最穩健的做法，使用者等待時間可透過非同步查詢優化。

---

### Q4：內部 AI 助理和外部客服機器人的架構有何本質差異？

**重點回答：**

| 面向 | 外部客服機器人 | 內部員工助理 |
|------|--------------|-------------|
| **信任模型** | 零信任（用戶匿名） | 高信任（員工SSO驗證）|
| **知識庫** | 公開產品文件 | 含機密的內部文件 |
| **RBAC** | 不需要或簡單 | 複雜多層次 |
| **個人化** | 有限（訂單查詢）| 深度（個人假期、職位相關）|
| **工具整合** | 訂單/FAQ系統 | Jira/Slack/HR系統 |
| **擴展性挑戰** | 高並發、低延遲 | 長上下文、深度查詢 |
| **合規要求** | GDPR（用戶資料）| 勞動法規、商業機密 |
| **失敗處理** | 轉人工客服 | 轉HR/IT/主管 |

**本質差異**：外部機器人解決客戶服務規模問題；內部機器人解決知識管理效率問題，兩者 ROI 評估方式完全不同。

---

### Q5：如何衡量內部 AI 助理的成功？KPI 應該設什麼？

**重點回答：**

**採用率指標（Adoption Metrics）：**
- 月活躍用戶（MAU）/ 全體員工比例（目標：6個月內達到 60%+）
- 每位員工月均查詢次數（目標：每月 50+ 次表示深度使用）
- 部門覆蓋率（是否各部門都有使用）

**效率指標（Efficiency Metrics）：**
- 平均問題解決時間（vs. 傳統方式如找HR、查Wiki）
- HR 重複性問題詢問量下降比例（目標：下降 40%）
- IT Helpdesk 工單數量變化

**品質指標（Quality Metrics）：**
- 用戶滿意度評分（每次對話後收集）
- 回答準確率（抽樣人工審查）
- 幻覺（Hallucination）發生率

**商業價值指標（Business Impact）：**
- 估算節省的人力時間（次數 × 平均節省分鐘 × 人力成本）
- 新人入職時間縮短（Onboarding Days to Productivity）
- 員工滿意度（ENPS 分數）

**反指標（Anti-Metrics，需監控）：**
- 安全事件次數（機密資料洩漏）
- 錯誤資訊傳播案例
- 過度依賴 AI 導致的判斷力下降

---
## 總結：企業內部 AI 助理系統設計要點

```
┌─────────────────────────────────────────────────────────────┐
│           企業內部 AI 助理系統設計要點總覽                     │
├─────────────────┬───────────────────────────────────────────┤
│ 安全性          │ RBAC + 機密等級分層 + 稽核記錄             │
│ 知識庫          │ 多來源向量庫 + 元資料過濾 + 定期更新       │
│ 個人化          │ 員工角色感知 + 部門情境 + 歷史記錄         │
│ 工具整合        │ Mock Jira + Slack + HR System              │
│ 生產力功能      │ 會議摘要 + Email助理 + 任務提取            │
│ 成本優化        │ 模型分級 + 語意快取 + 使用量配額           │
│ 成功衡量        │ 採用率 + 效率提升 + 員工滿意度             │
└─────────────────┴───────────────────────────────────────────┘
```

### 技術堆疊回顧

| 元件 | 技術選擇 | 說明 |
|------|---------|------|
| LLM | OpenAI gpt-4o-mini | 成本效益高，適合內部使用 |
| Embedding | text-embedding-3-small | 多語支援，成本低 |
| 向量資料庫 | ChromaDB In-Memory | 原型快速開發，生產建議 Persistent |
| 身份驗證 | SSO（模擬）| 實際部署需整合 AD/Okta |
| 工具整合 | OpenAI Function Calling | 結構化工具呼叫 |
| 框架 | 純 Python（無 LangChain）| 完整掌控邏輯 |

### 下一步學習

- **NB15**：生產環境部署 — Docker、K8s、CI/CD 最佳實踐
- **NB16**：向量資料庫選型 — Pinecone vs. Weaviate vs. Qdrant vs. ChromaDB
- **NB17**：RAG 系統的持續優化 — A/B 測試、回饋循環、知識庫自動更新